In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style = "color:red"> ch02_ LLM 활용의 기본 개념(Ollama)</span>

# 1. LLM을 활용하여 답변 생성하기

## 1) Ollama의 로컬 LLM 이용
성능은 GPT, Claude 같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ① ollama.com 다운로드 -> 설치 -> 모델 pull
cmd, powershell -> ollama pull <모델이름> ex) ollama pull deepseek-r1:1.5b(window키 + R => powershell)
- llama 공식적으로 한글 지원 안됨, 그러나 llama3.1:405b는 한글을 지원 가능, -> llama3.3 70b또한 가능)
- exaone 공식적으로 한글 지원 (LG)

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="deepseek-r1:1.5b")

In [5]:
result = llm.invoke('한국의 수도는 어디에요?')
result # 추론모델 <thinking>

AIMessage(content="<think>\n\n</think>\n\n한국의 도 resides에서 경제적 기여와 인력과 전망을 combination does make a big difference. According to the World Bank's 2014 Global Development Report, the South to North Sea region in China contributes around 3.8% of global GDP. This area is characterized by a strong emphasis on agriculture (such as rice production), tourism, and industry, which has historically supported its economic growth.", additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-25T02:11:51.2310162Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5138251800, 'load_duration': 27745200, 'prompt_eval_count': 12, 'prompt_eval_duration': 257363900, 'eval_count': 93, 'eval_duration': 4851613800, 'model_name': 'deepseek-r1:1.5b'}, id='run--70db142f-83cb-4d10-a231-739a599080ba-0', usage_metadata={'input_tokens': 12, 'output_tokens': 93, 'total_tokens': 105})

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model = 'llama3.2:1b')
result = llm.invoke("What is the capital of South Korea?")
result.content

'The capital of South Korea is Seoul.'

In [13]:
llm.invoke('한국의 수도는 어디야?')

AIMessage(content='한국의 수도는 서울입니다.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T02:19:59.202299Z', 'done': True, 'done_reason': 'stop', 'total_duration': 515482000, 'load_duration': 30494100, 'prompt_eval_count': 32, 'prompt_eval_duration': 60469700, 'eval_count': 8, 'eval_duration': 422919700, 'model_name': 'llama3.2:1b'}, id='run--806cbe17-58be-47d2-95c3-ac89f9e1ea3f-0', usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

## 2) openai 활용
- pip install langchain-openai

In [16]:
from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model = 'gpt-4o-mini',)
# result = llm.invoke("What is the capital of South Korea?")
# result.content => 에러 이유 : OPENAI_API_KEY 환경변수의 부재

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [17]:
# 환경변수 가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [19]:
# 코랩에서 OPENAI_API_KEY 읽어오기(.env못씀)
# 보안키 추가 후 
# from google.colab import userdata
# userdata.get('secretName') # secretName => 지정한 보안키 이름 사용

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano',
                openai_api_key = os.getenv('TEST_KEY'),
                )
result = llm.invoke("What is the capital of Korea?")

In [22]:
result.content

'The capital of South Korea is Seoul.'

In [25]:
# 모든 모델의 키가 OPENAI_API_KEY는 아님 -> 보통의 경우 openAI의 모델은 .env 속 api키 이름을 OPENAI_API_KEY 로 사용하면 openai_api_key 지정이 필요 없음
# Claude -> Anthropic
# Azure, upstage, Bedrock : 에러 메세지 참조하여 환경변수 생성

In [26]:
# from langchain_openai import AzureOpenAI

# llm = AzureOpenAI(model='gpt-4.1-nano')
# llm.invoke("What is the capital of Korea?")

# 에러를 내면 OPENAI_API_VERSION 환경 변수가 필요하다는 메세지

In [28]:
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model = 'claude-3-5-sonnet-20240620')
# llm.invoke("What is the capital of Korea?")
# # 에러 메세지를 봐도 환경 변수 이름을 알 수 없음
# # ChatAnthropic를 검색 후 Docs 사이트에서 명시한 ANTHROPIC_API_KEY를 .env에 환경변수로 추가해주어야 한다.

# 2. Langchain 스타일로 프롬프트 작성하기
- 프롬프트 : llm 호출 시 쓰는 질문

In [31]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0)
# 프롬프트 타입 : PromptValue, str, or list of BaseMessages

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate을 사용하여 변수가 포함된 템플릿 작성

In [36]:
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model='llama3.2:1b')
prompt_template = PromptTemplate(
    template = "What is the capital of {country}?", # {}안의 값에 새로운 값을 대입하여 사용
    input_variables = ['country']
)
prompt = prompt_template.invoke({'country':'Korea'})
print(prompt)
llm.invoke(prompt)

text='What is the capital of Korea?'


AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T05:31:53.3392073Z', 'done': True, 'done_reason': 'stop', 'total_duration': 679708300, 'load_duration': 23630300, 'prompt_eval_count': 32, 'prompt_eval_duration': 155580500, 'eval_count': 9, 'eval_duration': 499987800, 'model_name': 'llama3.2:1b'}, id='run--7a0940b8-8bbe-4597-a292-a84a3adeee0a-0', usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41})

## 2) 메세지 기반 프롬프트 작성
list of BaseMessages
- BaseMessage 리스트
- BaseMessage 상속 받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage

In [41]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
message_list = [
    SystemMessage(content = "You're a expert of Republic of Korea!"),
    HumanMessage(content = "What is the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content = "What is the capital of Korea?"),
    AIMessage(content="The capital of Italy is Seoul."),
    HumanMessage(content = "What is the capital of France?")
]
llm.invoke(message_list)

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T05:51:42.7656387Z', 'done': True, 'done_reason': 'stop', 'total_duration': 515751700, 'load_duration': 29081800, 'prompt_eval_count': 89, 'prompt_eval_duration': 61420100, 'eval_count': 8, 'eval_duration': 424226600, 'model_name': 'llama3.2:1b'}, id='run--3a7398f0-0337-4c8e-b08d-6f495c19c019-0', usage_metadata={'input_tokens': 89, 'output_tokens': 8, 'total_tokens': 97})

In [43]:
# BaseMessage lisf로 하면 랭체인화 X, ChatPromptTemplate X
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
message_list = [
    SystemMessage(content = "You're a expert of Republic of Korea!"),
    HumanMessage(content = "What is the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content = "What is the capital of Korea?"),
    AIMessage(content="The capital of Italy is Seoul."),
    HumanMessage(content = "What is the capital of {country}?")
]
from langchain_core.prompts import ChatPromptTemplate
chat_prompt_template = ChatPromptTemplate.from_messages(message_list) # 리스트 프롬프트화
prompt = chat_prompt_template.invoke({'country' : "Korea"})
print(prompt)

messages=[SystemMessage(content="You're a expert of Republic of Korea!", additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Seoul.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of {country}?', additional_kwargs={}, response_metadata={})]


## 3) ChatPromptTemplate 사용
- BaseMessage 리스트 -> 튜플 리스트

### 랭체인을 위해서는 아래와 같은 방식으로 코드 필요

In [56]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant"),
    ("human", "What is the capital of Italy?"), # human 대시 user도 가능
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of Korea?"),
    ("ai", "The capital of Korea is Seoul."),
    ("user", "What is the capital of {country}?")
])
country = input('어느 나라의 수도가 궁금하세요? : ')
prompt = chat_prompt_template.invoke({'country': country})
print("프롬프트",prompt)
llm.invoke(prompt)

어느 나라의 수도가 궁금하세요? : 대민한국
프롬프트 messages=[SystemMessage(content='You are helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of 대민한국?', additional_kwargs={}, response_metadata={})]


AIMessage(content='I think there may be a misunderstanding. The term "대민 한국" (Dai Min Hakunan) doesn\'t appear to be a widely recognized or established term for a country in South Korea.\n\nHowever, I believe you may be referring to North Korea, which is also known as the Democratic People\'s Republic of Korea (DPRK). If that\'s the case, the capital of North Korea is Pyongyang.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T06:20:05.6719811Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5524372300, 'load_duration': 27912500, 'prompt_eval_count': 86, 'prompt_eval_duration': 198157300, 'eval_count': 84, 'eval_duration': 5295583200, 'model_name': 'llama3.2:1b'}, id='run--08d3089d-c601-4dd5-917e-1654ad5a3364-0', usage_metadata={'input_tokens': 86, 'output_tokens': 84, 'total_tokens': 170})

In [60]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 대한민국 전문가입니다."),
    ("human", "이태리의 수도는 어디인가요?"), # human 대시 user도 가능
    ("ai", "이태리의 수도는 로마입니다."),
    ("human", "한국의 수도는 어디인가요?"),
    ("ai", "한국의 수도는 서울입니다."),
    ("user", "{country}의 수도는 어디인가요?")
])
country = input('어느 나라의 수도가 궁금하세요? : ')
prompt = chat_prompt_template.invoke({'country': country})
print("프롬프트",prompt)
llm.invoke(prompt) # 한글은 조금 모자라네

어느 나라의 수도가 궁금하세요? : 일본
프롬프트 messages=[SystemMessage(content='당신은 대한민국 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='이태리의 수도는 어디인가요?', additional_kwargs={}, response_metadata={}), AIMessage(content='이태리의 수도는 로마입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='한국의 수도는 어디인가요?', additional_kwargs={}, response_metadata={}), AIMessage(content='한국의 수도는 서울입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='일본의 수도는 어디인가요?', additional_kwargs={}, response_metadata={})]


AIMessage(content='일본의 수도는 도호라카와 대신, 오이타시와 요코다 마에시에서 동시에 운영하는 수도가 있습니다.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T06:22:55.4533389Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2286047100, 'load_duration': 27114200, 'prompt_eval_count': 94, 'prompt_eval_duration': 282685300, 'eval_count': 33, 'eval_duration': 1974682600, 'model_name': 'llama3.2:1b'}, id='run--21579595-0776-4294-b18d-4b7c8bd89a44-0', usage_metadata={'input_tokens': 94, 'output_tokens': 33, 'total_tokens': 127})

# 3. 답변 형식을 컨트롤하기
- invoke 실행결과는 AIMessage() -> String이거나 Json, 객체 : outputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 사용하여 LLM출력(AIMessage)을 단순 문자열로 변환

In [67]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template = "What is the capital of {country}? Return the name of the city only." ,
    input_variables = ['country']
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({'country':'Korea'})
print(prompt)
result = llm.invoke(prompt) # AIMessage객체
print('LLM 결과 :', type(result), result.content)
# 문자열 출력 Parser를 이용해서 LLM 응답을 단순 문자열로 변환
output_parser = StrOutputParser()
print("parser 결과 :",output_parser.invoke(result))

text='What is the capital of Korea? Return the name of the city only.'
LLM 결과 : <class 'langchain_core.messages.ai.AIMessage'> Seoul
parser 결과 : Seoul


In [69]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

'Seoul'

In [76]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')

# PromptTemplate(변수설정) => ChatPromptTemplate(변수설정, system, user, ai 지정을 통한 모범답안 지정 가능)
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant with experties in South Korea."),
#     ("human", "What is the capital of Italy?"), # human 대시 user도 가능
#     ("ai", "Rome."),
#     ("human", "What is the capital of Korea?"),
#     ("ai", "Seoul."),
    ("user", "What is the capital of {country}? Return the name of the city only")
])

output_parser = StrOutputParser()
# country = input('어느 나라의 수도가 궁금하세요? : ')

prompt = chat_prompt_template.invoke({'country': 'Korea'})

output_parser.invoke(llm.invoke(prompt)) # 한글은 조금 모자라네

'Seoul'

## 2) Json 출력 파서 이용
- json()으로 응답하기를 원하지만 우선 어떤 형식으로 반환되는 지 확인

{"name" : "홍","age" : 22}(json) / {'name' : '홍','age' : 22}(dict)

In [83]:
from langchain_core.output_parsers import JsonOutputParser
count_detail_prompt = PromptTemplate(
                                    template="""Give following information about {country}
                                    - Capital
                                    - Population
                                    - Language
                                    - Currency
                                    return it is JSON format and return JSON dictionary only""",
                                    input_variables=['country']
                                    )
prompt = count_detail_prompt.invoke({'country':'Korea'})
print(type(prompt))

ai_message = llm.invoke(prompt)
print(type(ai_message))

json_output_parser = JsonOutputParser()
data_dict = json_output_parser.invoke(ai_message)

print(data_dict, type(data_dict))

<class 'langchain_core.prompt_values.StringPromptValue'>
<class 'langchain_core.messages.ai.AIMessage'>
{'capital': 'Seoul', 'population': 50800000, 'language': 'Korean', 'currency': 'Won (KRW)'} <class 'dict'>


In [86]:
count_detail_prompt = PromptTemplate(
                                    template="""Give following information about {country}
                                    - Capital
                                    - Population
                                    - Language
                                    - Currency
                                    return it is JSON format and return JSON dictionary only""",
                                    input_variables=['country']
                                    )
output_parser = JsonOutputParser()
info = output_parser.invoke(llm.invoke(count_detail_prompt.invoke({'country' : 'Korea'})))
print(type(info))

<class 'dict'>


## 3) 구조화된 class 출력 사용
- Pydantic 모델을 사용하여 LLM 출력을 구조화된 형식으로 받기(JsonOutputParser 보다 훨씬 안정적)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리

In [101]:
# 일반적인 class
class User : 
    def __init__(self, id, name, is_active=True) :
        self.id = id
        self.name = name
        self.is_active = is_active
        if len(self.name) <=1 :
            raise Exception
#     def __str__(self) :
#         return self.name + " " + self.id
user = User(1, '1동')
print(user)

In [102]:
# 기본 개념
from pydantic import BaseModel, Field
class User(BaseModel) :
    id : int
    name : str
    is_active : bool=True
# user = User(id="a",name='홍길동') # X => id int화 불가능하여 에러 발생
user = User(id="1",name='홍길동') # ok => id 자동 int
print(user)

id=1 name='홍길동' is_active=True


In [108]:
from pydantic import BaseModel, Field
class User(BaseModel) :
    id : int = Field(gt=0, description = 'id') # gt=0 : id>0, ge : id >=0, lt : id<0, le : id <=0
    name : str = Field(min_length=2, description = 'name')
    is_active : bool=Field(default=True, description = 'id활성화')
# user = User(id="a",name='홍길동') # X => id int화 불가능하여 에러 발생
user = User(id="1",name='홍길동') # ok => id 자동 int
print(user)

id=1 name='홍길동' is_active=True


In [122]:
count_detail_prompt = PromptTemplate(
                                    template="""Give following information about {country}
                                    - Capital
                                    - Population
                                    - Language
                                    - Currency
                                    return it is JSON format and return JSON dictionary only""",
                                    input_variables=['country']
                                    )
class CountryDetail(BaseModel) : # description : 더 정확한 출력을 유도
    capital : str = Field(description="the capital of the country")
    population : str = Field(description="the population of the country")
    language : str = Field(description="the language of the country")
    currency : str = Field(description="the currency of the country")
# 출력 형식 파서 + LLM
Structuredllm = llm.with_structured_output(CountryDetail)

# 기존 방식
# output_parser = JsonOutputParser()
# output_parser.invoke(llm.invoke(count_detail_prompt.invoke({'country':'Korea'})))

info = Structuredllm.invoke(count_detail_prompt.invoke({'country':'korea'}))
type(info), info.capital, info.population, info.language, info.currency

(__main__.CountryDetail,
 'Seoul',
 'Our estimated population in 2021 was approximately 51.8 million.',
 'Korean (official)',
 'South Korean won (KRW)')

In [123]:
print(info)
print(info.capital)
print(info.population)
print(info.language)
print(info.currency)
# print('info를 json :',info.json())
print('info를 json :',info.model_dump_json())
# print('info를 dict :',info.dict())
print('info를 dict :',info.model_dump())

capital='Seoul' population='Our estimated population in 2021 was approximately 51.8 million.' language='Korean (official)' currency='South Korean won (KRW)'
Seoul
Our estimated population in 2021 was approximately 51.8 million.
Korean (official)
South Korean won (KRW)
info를 json : {"capital":"Seoul","population":"Our estimated population in 2021 was approximately 51.8 million.","language":"Korean (official)","currency":"South Korean won (KRW)"}
info를 dict : {'capital': 'Seoul', 'population': 'Our estimated population in 2021 was approximately 51.8 million.', 'language': 'Korean (official)', 'currency': 'South Korean won (KRW)'}


# 4. LCEL을 활용한 랭체인 생성하기
## 1) 문자열 출력 파서 사용
- invoke

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

llm = ChatOllama(model = 'llama3.2:1b',
                 temperature = 0, # 일관된 답변(창의적이지 않은)
                )
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template = "What is the capital of {country}? Return the name of the city only." ,
    input_variables = ['country']
)
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

'Seoul'

## 2) LCEL을 사용한 간단한 체인 구성
- pipe 연산자(|) 사용

In [4]:
# prompt => llm => output_parser 연결하는 체인 생성
capital_chain = prompt_template | llm | output_parser

# 생성된 체인 invoke
capital_chain.invoke({'country':'Korea'})

'Seoul'

In [5]:
type(capital_chain) # langchain_core.runnables.base.RunnableSequence

langchain_core.runnables.base.RunnableSequence

## 3) 복합 체인 구성
- 여러 단계의 추론이 필요한 경우 (체인 연결)

In [8]:
# 나라 설명 -> 나라
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

prompt_template = PromptTemplate(
                                template="""Guess the name of country based on the following information:
                                {information}.
                                Return the name of the country only.""",
                                input_variable = ["information"],
                                )
llm = ChatOllama(model = 'llama3.2:1b',
                 temperature = 0, # 일관된 답변(창의적이지 않은)
                )
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({'information' : "This country is very famous for its wine"})))

'Italy.'

In [19]:
# 나라명 추측 체인 생성
country_chain = prompt_template | llm | output_parser
# type(country_chain) # langchain_core.runnables.base.RunnableSequence
country_chain.invoke({'information' : 'This country is really famous for soju'})

'South Korea.'

In [23]:
# 나라 설명 -> (나라명 -> ) 그 나라 수도
prompt_template2 = PromptTemplate(template = "What is the capital of {country}? Return the name of country and capital only with format the name of country/the name of capital of this country.", input_variable = ["country"])
capital_chain = prompt_template2|llm|output_parser
chain = country_chain|capital_chain
chain.invoke({'information':'This country is very famous for its soju'}) # information을 적는 것 마저 귀찮아진다면..?

'South Korea / Seoul'

In [25]:
# chain.invoke('This country is very famous for its soju') # 이렇게 작성하도록 유도
chain = {'country':country_chain} | capital_chain
chain.invoke({'information':'This country is very famous for its soju'}) # information을 적는 것 마저 귀찮아진다면..?

'South Korea / Seoul'

In [31]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {'information':RunnablePassthrough()}|{'country':country_chain}|capital_chain
final_chain.invoke("This country is very famous for its soju")

'South Korea / Seoul'

In [32]:
# 프롬프트 템플릿에 변수 2
# 나라설명 -> 나라
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
                                template="""Guess the name of country in the {continent} based on the following information:
                                {information}.
                                Return the name of the country only.""",
                                input_variable = ["information","continent"],
                                )
# output_parser.invoke(llm.invoke(prompt_template.invoke({'continent':"","information":""})))
country_chain = prompt_template|llm|output_parser
country_chain.invoke({'continent':"Europe",
                      "information":"This country is very famous for its wine"})

'Italy.'

In [33]:
final_chain = {"country" : country_chain} | capital_chain
final_chain.invoke({'continent':"Europe",
                      "information":"This country is very famous for its wine"})

'Italy. Rome'